# SenCultureAge

## Index
1. [Instantiate model class](#Instantiate-model-class)
2. [Define clock metadata](#Define-clock-metadata)
3. [Download clock dependencies](#Download-clock-dependencies)
5. [Load features](#Load-features)
6. [Load weights into base model](#Load-weights-into-base-model)
7. [Load reference values](#Load-reference-values)
8. [Load preprocess and postprocess objects](#Load-preprocess-and-postprocess-objects)
10. [Check all clock parameters](#Check-all-clock-parameters)
10. [Basic test](#Basic-test)
11. [Save torch model](#Save-torch-model)
12. [Clear directory](#Clear-directory)

Let's first import some packages:

In [1]:
import os
import inspect
import shutil
import json
import torch
import pandas as pd
import pyaging as pya

## Instantiate model class

In [2]:
def print_entire_class(cls):
    source = inspect.getsource(cls)
    print(source)

print_entire_class(pya.models.SenCultureAge)

class SenCultureAge(LinearReferenceClock):
    pass



In [3]:
model = pya.models.SenCultureAge()

## Define clock metadata

In [4]:
model.metadata["clock_name"] = "sencultureage"
model.metadata["data_type"] = "DNA methylation"  # Paper: The predictors use CpG DNA-methylation beta values.
model.metadata["species"] = "Homo sapiens"  # Paper: All three assigned predictors were developed from human cell or human whole-blood datasets.
model.metadata["year"] = 2026
model.metadata["approved_by_author"] = "⌛"
model.metadata["citation"] = "Kasamoto, K., Gibson, J., Moqri, M., Smith, R. & Higgins-Chen, A.T. DNA methylation signatures of cellular senescence are not reversed by senolytic treatment. Aging Cell 25, e70430 (2026)."
model.metadata["doi"] = "https://doi.org/10.1111/acel.70430"
model.metadata["notes"] = "Binomial elastic-net classifier of in-vitro cellular senescence, trained after ComBat correction on pooled human fibroblast and mesenchymal-stromal-cell datasets and restricted to direction-concordant senescence/age/mortality CpGs."
model.metadata["research_only"] = None
model.metadata["tissue"] = ["cultured fibroblasts", "cultured mesenchymal stromal cells"]  # Paper: Training pooled fibroblasts, diploid skin fibroblasts, bone-marrow MSCs and adipose-derived MSCs.
model.metadata["predicts"] = ["cellular senescence"]  # Paper: SenCultureAge is a binary senescence predictor.
model.metadata["training_target"] = ["cellular senescence"]  # Paper: Each in-vitro training sample was labeled senescent or control.
model.metadata["unit"] = ["log odds"]  # Paper: Pyaging applies the binomial-model coefficients without an inverse-logit postprocess.
model.metadata["model_type"] = "elastic net logistic regression"  # Paper: The model used glmnet with family binomial and alpha 0.5.
model.metadata["platform"] = ["Illumina 450K", "Illumina EPIC"]  # Paper: Feature discovery and predictor training used human 450K and EPIC methylation datasets.
model.metadata["population"] = "human cell cultures"  # Paper: Training pooled GSE197723 and GSE227160 senescent and control cultures.
model.metadata["journal"] = "Aging Cell"
model.metadata["last_author"] = "Albert T. Higgins-Chen"
model.metadata["n_features"] = 142
model.metadata["citations"] = 0
model.metadata["citations_date"] = "2026-07-05"


## Download clock dependencies

In [5]:
os.system(f"curl -sL -o SenCultureAge_CpGs.csv https://raw.githubusercontent.com/HigginsChenLab/methylCIPHER/19b12296b0d7eb7055a97d068064df635f44ce3e/data-raw/SenescenceAge/SenCultureAge_CpGs.csv")

0

## Load features

In [6]:
coef_df = pd.read_csv('SenCultureAge_CpGs.csv')
model.features = coef_df['CpG'].tolist()

## Load weights into base model

In [7]:
weights = torch.tensor(coef_df['Coefficient'].tolist()).unsqueeze(0).float()
intercept = torch.tensor([-254.6817]).float()

In [8]:
base_model = pya.models.LinearModel(input_dim=len(model.features))

base_model.linear.weight.data = weights.float()
base_model.linear.bias.data = intercept.float()

model.base_model = base_model

## Load reference values

In [9]:
model.reference_values = None

## Load preprocess and postprocess objects

In [10]:
model.preprocess_name = None
model.preprocess_dependencies = None

In [11]:
model.postprocess_name = None
model.postprocess_dependencies = None

## Check all clock parameters

In [12]:
pya.utils.print_model_details(model)


%==================================== Model Details ====================================%
Model Attributes:

training: True
metadata: {'approved_by_author': '⌛',
 'citation': 'Kasamoto, Kotaro, et al. "DNA methylation clocks for estimating '
             'replicative senescence in human cells." Aging Cell (2026): '
             'e70430.',
 'clock_name': 'sencultureage',
 'data_type': 'methylation',
 'doi': 'https://doi.org/10.1111/acel.70430',
 'notes': None,
 'research_only': None,
 'species': 'Homo sapiens',
 'version': None,
 'year': 2026}
reference_values: None
preprocess_name: None
preprocess_dependencies: None
postprocess_name: None
postprocess_dependencies: None
features: ['cg00127591', 'cg00296189', 'cg00518989', 'cg00666978', 'cg00899883', 'cg01587390', 'cg01779934', 'cg01788025', 'cg02389682', 'cg02484633', 'cg03063057', 'cg03917020', 'cg04081392', 'cg04224041', 'cg04514998', 'cg04796775', 'cg04842828', 'cg05076775', 'cg05198969', 'cg05275153', 'cg06955158', 'cg07166216', 'c

## Basic test

In [13]:
torch.manual_seed(42)
input = torch.randn(10, len(model.features), dtype=float)
model.eval()
model.to(float)
pred = model(input)
pred

tensor([[  77.5496],
        [-329.8605],
        [-197.1059],
        [-396.4821],
        [-519.2351],
        [-122.0558],
        [-104.6654],
        [-262.1782],
        [-182.1105],
        [-254.5931]], dtype=torch.float64, grad_fn=<AddmmBackward0>)

## Save torch model

In [14]:
torch.save(model, f"../weights/{model.metadata['clock_name']}.pt")

## Clear directory
<a id="10"></a>

In [15]:
# Function to remove a folder and all its contents
def remove_folder(path):
    try:
        shutil.rmtree(path)
        print(f"Deleted folder: {path}")
    except Exception as e:
        print(f"Error deleting folder {path}: {e}")

# Get a list of all files and folders in the current directory
all_items = os.listdir('.')

# Loop through the items
for item in all_items:
    # Check if it's a file and does not end with .ipynb
    if os.path.isfile(item) and not item.endswith('.ipynb'):
        os.remove(item)
        print(f"Deleted file: {item}")
    # Check if it's a folder
    elif os.path.isdir(item):
        remove_folder(item)

Deleted file: SenCultureAge_CpGs.csv
